# 🔤 Lekcja 6: Jak nauczyć komputer czytać słowa? (Cechy porządkowe, nominalne, Ordinal i One-Hot Encoding)

Do tej pory nasz model korzystał wyłącznie z **8 kolumn liczbowych** (takich jak metraż `GrLivArea` czy rok budowy `YearBuilt`). Ale w naszej tabeli `../data/train.csv` aż **43 kolumny zawierają tekst (słowa)**! Są tam informacje o dzielnicy (`Neighborhood`), jakości kuchni (`KitchenQual`), klimatyzacji (`CentralAir`) czy rodzaju dachu.

Gdybyś spróbował wrzucić kolumnę ze słowami `'Ex'`, `'Gd'`, `'TA'` bezpośrednio do modelu `model.fit()`, Python natychmiast zatrzymałby się z czerwonym błędem:  
`ValueError: could not convert string to float: 'Ex'` (*„Nie potrafię zamienić napisu `'Ex'` na liczbę!”*).

--- 

### 🧠 Złota zasada Data Science: Nie każdy tekst zamienia się na liczby tak samo!

Zanim zamienimy słowa na liczby (po angielsku ten proces nazywa się **Encoding**, czyli **kodowanie**), musimy podzielić kolumny tekstowe na **dwie zupełnie różne rodziny**:

| Rodzina cech tekstowych | Życiowy przykład | Przykład z naszych domów (`train.csv`) | Czy istnieje tu naturalna drabina (od najmniejszego do największego)? | Jak zamieniamy je na liczby? |
| :--- | :--- | :--- | :--- | :--- |
| **1. Cechy PORZĄDKOWE** *(Ordinal Features)* | • Rozmiar koszulki: `S < M < L < XL`<br>• Wykształcenie: `Podstawowe < Średnie < Wyższe`<br>• Stopnie wojskowe | **`KitchenQual`** (jakość kuchni), **`ExterQual`** (jakość elewacji), **`BsmtQual`** (jakość piwnicy), **`FireplaceQu`** (jakość kominka):<br>`Brak < Po < Fa < TA < Gd < Ex` | **TAK!** Doskonała kuchnia (`Ex`) jest obiektywnie lepsza od dobrej (`Gd`), a dobra jest lepsza od przeciętnej (`TA`). | **Ordinal Encoding (Kodowanie porządkowe):** Przypisujemy słowa do kolejnych stopni drabiny w jednej kolumnie: **`0, 1, 2, 3, 4, 5`** (za pomocą słownika i metody `.map()`). |
| **2. Cechy NIEUPORZĄDKOWANE / NOMINALNE** *(Nominal Features)* | • Kolor oczu: `Niebieskie, Zielone, Brązowe`<br>• Kraj: `Polska, Francja, Japonia`<br>• Marka auta | **`Neighborhood`** (25 dzielnic miasta, np. `NoRidge`, `OldTown`, `CollgCr`), **`GarageType`** (typ garażu), **`Foundation`** (materiał fundamentów) | **NIE!** Nazwy dzielnic czy materiałów to po prostu różne etykiety — nie tworzą jednej równej miarki od 1 do 25. | **One-Hot Encoding (Kodowanie zero-jedynkowe):** Rozbijamy kolumnę na **tablicę przełączników `0` / `1`** (za pomocą funkcji `pd.get_dummies()`). |

Zacznijmy od pierwszej rodziny — **cech porządkowych (*Ordinal Encoding*)**!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Wczytujemy dane z folderu ../data/
train = pd.read_csv('../data/train.csv')

# Zobaczmy, jakie słowa siedzą w kolumnie KitchenQual (jakość kuchni) oraz ile kosztują takie domy:
print('--- Średnia cena domu dla każdej oceny jakości kuchni (KitchenQual) ---')
print(train.groupby('KitchenQual')['SalePrice'].agg(['count', 'mean']).round(0))

# 2. Tworzymy nasz własny SŁOWNIK TŁUMACZĄCY (drabina od 0 do 5 zgodnie z data_description.txt):
#    Brak = 0 | Po (Poor/Słaba) = 1 | Fa (Fair/Dostateczna) = 2
#    TA (Typical/Przeciętna) = 3 | Gd (Good/Dobra) = 4 | Ex (Excellent/Doskonała) = 5
mapa_jakosci = {
    'Brak': 0,
    'Po': 1,
    'Fa': 2,
    'TA': 3,
    'Gd': 4,
    'Ex': 5
}

# 3. Tłumaczymy 4 ważne kolumny jakościowe ze słów na liczby za pomocą metody .map(mapa_jakosci)
kolumny_porzadkowe = ['KitchenQual', 'ExterQual', 'BsmtQual', 'FireplaceQu']

for kol in kolumny_porzadkowe:
    # Najpierw zamieniamy ewentualne puste kratki NaN na słowo 'Brak', a potem tłumaczymy słownikiem!
    train[kol + '_Num'] = train[kol].fillna('Brak').map(mapa_jakosci)

# 4. Porównajmy stare kolumny tekstowe i nowe kolumny liczbowe (_Num) dla pierwszych 5 domów:
print('\n--- Porównanie PRZED (tekst) i PO (liczby 0-5) dla pierwszych 5 domów ---')
train[['KitchenQual', 'KitchenQual_Num', 'FireplaceQu', 'FireplaceQu_Num']].head(5)

### 🔬 Co dzieje się pod maską w `.map(mapa_jakosci)` i dlaczego nie wolno numerować słów alfabetycznie?

#### 1. Dlaczego stworzyliśmy słownik `mapa_jakosci` ręcznie, zamiast kazać komputerowi ponumerować słowa automatycznie?
Spójrz na pierwszą tabelkę z wydruku powyżej:
* Kuchnia **`Fa`** (*Fair* — dostateczna): średnia cena domu to **`105 565 $`**
* Kuchnia **`TA`** (*Typical* — przeciętna): średnia cena domu to **`139 963 $`**
* Kuchnia **`Gd`** (*Good* — dobra): średnia cena domu to **`212 116 $`**
* Kuchnia **`Ex`** (*Excellent* — doskonała): średnia cena domu to **`328 555 $`**

Gdybyśmy użyli bezmyślnego automatu, który nie zna języka angielskiego, automat posortowałby te skróty **alfabetycznie (od A do Z)**:
* `'Ex'` (zaczyna się na **E**) dostałoby numer **`1`**,
* `'Fa'` (na **F**) dostałoby numer **`2`**,
* `'Gd'` (na **G**) dostałoby numer **`3`**,
* `'TA'` (na **T**) dostałoby numer **`4`**!

Widzisz tę katastrofę?! Alfabetycznie luksusowa kuchnia `'Ex'` dostałaby najniższą ocenę `1`, a przeciętna `'TA'` najwyższą ocenę `4` — drabinę wywrócono by do góry nogami!  
Dlatego przy cechach porządkowych **zawsze sami układamy słownik w nawiasach klamrowych `{}`**: `'Brak': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5`.

---

#### 2. Jak działa łańcuch `train[kol].fillna('Brak').map(mapa_jakosci)`?
* **Krok 1 (`.fillna('Brak')`):** W kolumnie `FireplaceQu` (jakość kominka) domy bez kominka mają wartość `NaN`. Zamieniamy ją na słowo `'Brak'`.
* **Krok 2 (`.map(mapa_jakosci)`):** Metoda `.map()` działa jak **tłumacz ze słownikiem w ręku**. Idzie wiersz po wierszu przez wszystkie 1460 domów:
  * Widzi w kratce napis `'Gd'` $\rightarrow$ szuka `'Gd'` w słowniku `mapa_jakosci` $\rightarrow$ wpisuje liczbę **`4`**.
  * Widzi napis `'TA'` $\rightarrow$ szuka w słowniku $\rightarrow$ wpisuje liczbę **`3`**.
  * Widzi napis `'Brak'` $\rightarrow$ szuka w słowniku $\rightarrow$ wpisuje liczbę **`0`**.
* **`train[kol + '_Num']`:** W Pythonie znak `+` między dwoma tekstami skleja je ze sobą! Jeśli `kol` to `'KitchenQual'`, to `kol + '_Num'` tworzy nową nazwę kolumny `'KitchenQual_Num'`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Zobaczmy, jak działa One-Hot Encoding (pd.get_dummies) na przykładzie klimatyzacji (CentralAir) i dzielnicy (Neighborhood)
przyklad_ohe = pd.get_dummies(train[['CentralAir', 'Neighborhood']], dtype=int)
print(f'Przed One-Hot Encodingiem: 2 kolumny tekstowe.')
print(f'Po One-Hot Encodingu (pd.get_dummies): {przyklad_ohe.shape[1]} kolumn zero-jedynkowych (0 lub 1)!\n')
print('Podgląd pierwszych 6 przełączników (0/1) dla 5 pierwszych domów:')
print(przyklad_ohe.iloc[:5, :6])

# 2. WIELKI EKSPERYMENT: Sprawdźmy, jak dodawanie kolejnych grup cech obniża błąd MAE naszego modelu!
y = train['SalePrice']

# Zestaw A: 5 cech liczbowych z Lekcji 4
cechy_5 = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'YearBuilt']

# Zestaw B: 8 cech liczbowych z końca Lekcji 4 / Lekcji 5
cechy_8 = cechy_5 + ['YearRemodAdd', 'LotArea', 'BsmtFinSF1']

# Zestaw C: 8 cech liczbowych + 4 zakodowane porządkowo cechy jakościowe (Ordinal Encoding)
cechy_12_ordinal = cechy_8 + ['KitchenQual_Num', 'ExterQual_Num', 'BsmtQual_Num', 'FireplaceQu_Num']

# Zestaw D: Powyższe 12 cech + One-Hot Encoding dla dzielnicy (Neighborhood), klimatyzacji (CentralAir) i typu garażu (GarageType)
tabela_z_tekstem = train[cechy_12_ordinal + ['Neighborhood', 'CentralAir', 'GarageType']].fillna('Brak')
X_pelne_ohe = pd.get_dummies(tabela_z_tekstem, dtype=int)

# Trenujemy ten sam Las Losowy (100 drzew, random_state=42) na każdym z 4 zestawów i mierzymy błąd MAE w sejfie:
zestawy = [
    ('1. Tylko 5 liczb\n(Lekcja 4)', train[cechy_5]),
    ('2. 8 liczb\n(Lekcja 5)', train[cechy_8]),
    ('3. + Jakość 0-5\n(Ordinal Encoding)', train[cechy_12_ordinal]),
    ('4. + Dzielnice 0/1\n(One-Hot Encoding)', X_pelne_ohe)
]

nazwy_etapow = []
bledy_mae = []

for nazwa, X_aktualne in zestawy:
    X_tr, X_val, y_tr, y_val = train_test_split(X_aktualne, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    blad = mean_absolute_error(y_val, pred)
    nazwy_etapow.append(nazwa)
    bledy_mae.append(blad)

# 3. WIZUALIZACJA: Rysujemy dwa wykresy obok siebie!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Wykres lewy: Jak zakodowana jakość kuchni (2..5) wpływa na średnią cenę domu?
srednie_kuchnia = train.groupby('KitchenQual_Num')['SalePrice'].mean()
etykiety_kuchni = ['2: Fa (Słaba)', '3: TA (Przeciętna)', '4: Gd (Dobra)', '5: Ex (Doskonała)']
ax1.bar(etykiety_kuchni, srednie_kuchnia.values, color=['lightcoral', 'khaki', 'skyblue', 'mediumseagreen'], edgecolor='black')
ax1.set_title('1. Średnia cena domu wg jakości kuchni (Ordinal Encoding)')
ax1.set_xlabel('Zakodowana jakość kuchni (KitchenQual_Num)')
ax1.set_ylabel('Średnia cena domu ($)')
for i, v in enumerate(srednie_kuchnia.values):
    ax1.text(i, v + 5000, f'{v:,.0f} $', ha='center', fontweight='bold')
ax1.set_ylim(0, 370000)
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Wykres prawy: Spadek błędu MAE naszego modelu po dodaniu cech tekstowych!
kolory_etapow = ['lightgray', 'skyblue', 'gold', 'mediumseagreen']
slupki = ax2.bar(nazwy_etapow, bledy_mae, color=kolory_etapow, edgecolor='black', width=0.55)
ax2.set_title('2. Jak spada średni błąd modelu (MAE) dzięki nowym cechom?')
ax2.set_ylabel('Średni błąd wyceny MAE ($) — im MNIEJ, tym LEPIEJ!')
ax2.set_ylim(15000, 20500)
for i, v in enumerate(bledy_mae):
    ax2.text(i, v + 150, f'{v:,.0f} $', ha='center', fontweight='bold', fontsize=11)
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 💡 Jak działa One-Hot Encoding (`pd.get_dummies`) i dlaczego dzielnic NIE WOLNO numerować od 1 do 25?

#### 1. Dlaczego numerowanie dzielnic `1, 2, 3... 25` to matematyczna pułapka?
W mieście Ames jest 25 różnych dzielnic (`Neighborhood`), np. `'Blmngtn'`, `'CollgCr'`, `'NoRidge'`, `'OldTown'`.  
Co by się stało, gdybyśmy ponumerowali je w jednej kolumnie od `1` do `25` (tak jak zrobiliśmy z jakością kuchni)?

1. **Fałszywa drabina:** Dzielnica `'NoRidge'` (najdroższe luksusowe wille ze średnią ceną **335 295 \$**) dostałaby numer **`16`**, a dzielnica `'OldTown'` (stare, tanie domki ze średnią ceną **128 225 \$**) dostałaby numer **`18`**. Model patrzy na liczby i myśli: *„Skoro $18 > 16$, to `OldTown` jest bardziej luksusowe niż `NoRidge`!”*.
2. **Fałszywe dodawanie:** W matematyce $\frac{\text{Dzielnica } 1 + \text{Dzielnica } 17}{2} = \text{Dzielnica } 9$. Dla modelu liniowego oznaczałoby to, że średnia z dwóch przypadkowych dzielnic daje dzielnicę nr 9! (To tak, jakby powiedzieć: skoro `Polska = 1`, `Niemcy = 2`, a `Hiszpania = 3`, to `Polska + Niemcy = Hiszpania`!).

---

#### 2. Rozwiązanie: Tablica przełączników `0` / `1`, czyli One-Hot Encoding (`pd.get_dummies`)
Zamiast upychać różne dzielnice w jednej kolumnie na fałszywej drabinie, funkcja **`pd.get_dummies()`** robi coś genialnego — **rozbija jedną kolumnę tekstową na osobne kolumny-przełączniki (tak/nie = `1`/`0`) dla każdej dzielnicy**:

| Dom | Oryginalna kolumna `Neighborhood` | $\rightarrow$ `Neighborhood_CollgCr` | $\rightarrow$ `Neighborhood_NoRidge` | $\rightarrow$ `Neighborhood_OldTown` |
| :--- | :--- | :---: | :---: | :---: |
| **Dom 1** | `'CollgCr'` | **`1`** (Włączony!) | `0` | `0` |
| **Dom 2** | `'NoRidge'` | `0` | **`1`** (Włączony!) | `0` |
| **Dom 3** | `'OldTown'` | `0` | `0` | **`1`** (Włączony!) |

* **Skąd nazwa *One-Hot Encoding*?**  
  W elektronice włączony przewód z prądem (`1`) nazywa się *Hot* („gorący”), a wyłączony (`0`) — *Cold* („zimny”). W każdym wierszu tabeli dla danego domu **tylko JEDEN przełącznik dzielnicy jest włączony (`1` = *One-Hot*)**, a wszystkie pozostałe 24 dzielnice mają wartość `0`!
* **Dlaczego w `pd.get_dummies(..., dtype=int)` podajemy `dtype=int`?**  
  Domyślnie nowe wersje `pandas` wpisują do tych kolumn słowa `True` oraz `False`. Dodając `dtype=int` (*data type = integer*, czyli liczba całkowita), mówimy: *„Zapisz to jako czyste liczby `1` oraz `0`”*.

---

#### 3. Spójrz na prawy wykres powyżej — nasz model staje się coraz dokładniejszy!
Zobacz, jak systematycznie topnieje błąd wyceny **MAE** na tych samych 292 domach egzaminacyjnych z sejfu:
1. **5 cech liczbowych (Lekcja 4):** `19 180 $`
2. **8 cech liczbowych (Lekcja 5):** `18 380 $` *(błąd mniejszy o 800 \$)*
3. **+ 4 cechy porządkowe (Ordinal Encoding — kuchnia, elewacja, piwnica, kominek):** **`17 619 $`** *(kolejne -761 \$ błędu!)*
4. **+ Cechy nieuporządkowane (One-Hot Encoding — dzielnica, klimatyzacja, typ garażu):** **`17 451 $`** *(łącznie urwaliśmy już **1 729 \$** średniego błędu na każdym domu!)*

In [ ]:
# 🎯 ĆWICZENIE: Zakoduj samodzielnie dwie kolumny tekstowe (Ordinal oraz One-Hot Encoding)!
#
# 1. Ordinal Encoding ('HeatingQC' — jakość ogrzewania w skali 'Po', 'Fa', 'TA', 'Gd', 'Ex'):
#    Stwórz kolumnę `train['HeatingQC_Num']`, używając metody .map(mapa_jakosci) na kolumnie train['HeatingQC'],
#    i wyświetl pierwsze 5 wierszy: print(train[['HeatingQC', 'HeatingQC_Num']].head())
#
# 2. One-Hot Encoding ('Foundation' — rodzaj fundamentów budynku, bez naturalnej drabiny):
#    Stwórz zmienną `fundamenty_ohe = pd.get_dummies(train['Foundation'], dtype=int)`
#    i wyświetl pierwsze 5 wierszy tabeli `fundamenty_ohe` (.head())!

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Dwie rodziny cech tekstowych:** Cechy **porządkowe (*Ordinal*)** mają naturalną drabinę wartości (np. jakość kuchni `Brak < Po < Fa < TA < Gd < Ex`), natomiast cechy **nominalne (*Nominal*)** to równorzędne kategorie bez drabiny (np. dzielnica `Neighborhood` czy rodzaj fundamentów `Foundation`).
2. **Ordinal Encoding (`.map()`):** Cechy z naturalną hierarchią zamieniamy na kolejne stopnie liczbowe (`0, 1, 2, 3, 4, 5`) za pomocą własnego słownika i metody `.map(mapa_jakosci)` (nigdy alfabetycznie!).
3. **One-Hot Encoding (`pd.get_dummies(..., dtype=int)`):** Cechy bez naturalnej kolejności rozbijamy na osobne kolumny-przełączniki `0`/`1`, co zapobiega tworzeniu fałszywej hierarchii i obniża błąd naszego modelu w sejfie walidacyjnym do `17 451 $`.